# Εργαστήριο 7
Όπως και στα προηγούμενα εργαστήρια, πριν ξεκινήσετε εκτελέστε τα επόμενα 2 κελιά:

In [ ]:
interp.configureCompiler(_.settings.processArguments(List("-Wconf:cat=deprecation:s"), true))
interp.load.module(os.Path(s"${System.getProperty("user.dir")}/resource/chisel_deps.sc"))

In [ ]:
import chisel3._
import chisel3.util._
import chiseltest._
import chiseltest.RawTester.test

## 7.1 Προαπαιτούμενα modules
Για την υλοποίηση της τρέχουσας άσκησης θα χρειαστείτε **από τα προηγούμενα εργαστήρια** τα εξής:

* Το παραμετρικό module `Alu(n: Int)` (εργαστήριο 3, έχει παραδοθεί ως **Εργασία 1**)
* Το παραμετρικό module `RegisterFile(register_number: Int, register_width: Int)` (εργαστήριο 6, έχει παραδοθεί ως **Εργασία 2**)


### 7.1.1 Αντιγράψτε το module Alu
 **Αντιγράψτε** το module  `Alu` από το notebook που έχετε ήδη παραδόσει στο opencourses (**Εργασία 1, Άσκηση 2**) στο επόμενο κελί:

In [ ]:
class Alu(n: Int) extends Module {
  
  // συμπληρώστε τον κώδικα από το δικό σας module
  
}

**Ελέγξτε** το module `Alu`, εκτελώντας το επόμενο κελί:

In [ ]:
import scala.util.{Random => rnd }
try {
  test(new Alu(8)) { c =>
    for (i <- 0 until 10) {     // 10 φορές
      val a = rnd.nextInt(256)  // για τυχαίους αριθμούς a
      val b = rnd.nextInt(256)  // και b των 8 bits
      c.io.a.poke(a.U)
      c.io.b.poke(b.U)

      // test AND
      c.io.sel.poke(0.U)
      c.io.out.expect((a & b).U)
      // test OR
      c.io.sel.poke(1.U)
      c.io.out.expect((a | b).U)
      // test XOR
      c.io.sel.poke(2.U)
      c.io.out.expect((a ^ b).U)
      // test ADD/SUN
      c.io.sel.poke(3.U)
      // test ADD
      val result = a+b
      val sum = result & 0xFF  // απομάκρυνση κρατουμένου εξόδου από την τιμή ελέγχου
      c.io.sub.poke(0.U)
      c.io.out.expect(sum.U)
      // test SUB
      val result2 = a-b
      val sum2 = result2 & 0xFF  // απομάκρυνση κρατουμένου εξόδου από την τιμή ελέγχου
      c.io.sub.poke(1.U)
      c.io.out.expect(sum2.U)

    }
  }
  println("✅ SUCCESS!!")
} catch {
  case e: chisel3.internal.ChiselException => 
    println(s"❌ Chisel Elaboration Failed: ${e.getMessage}")
  case e: Throwable => 
    println(s"❌ Chisel Testing Failed: ${e.toString}")
}

### 7.1.2 Αντιγράψτε το module RegisterFile
 **Αντιγράψτε** το module  `RegisterFile` από το notebook που έχετε παραδόσει στο opencourses (**Εργασία 2**) στο επόμενο κελί:

In [ ]:
class RegisterFile(register_number: Int, register_width: Int) extends Module {
  
  // συμπληρώστε τον κώδικα από το δικό σας module
  
}

**Ελέγξτε** το module `RegisterFile`, εκτελώντας το επόμενο κελί:

In [ ]:
try {
  test(new RegisterFile(8,16)) { c =>

    for (i <- 0 to 7) {
      c.io.write_sel.poke(i.U)
      c.io.in.poke((333+i).U)
      c.clock.step()
    }
    
    for (i <- 0 to 7) {
      c.io.read_sel_a.poke(i.U)
      c.io.out_a.expect( if (i>0) (333+i).U else 0.U)
      c.io.read_sel_b.poke(i.U)
      c.io.out_b.expect( if (i>0) (333+i).U else 0.U)
    }

  }
  println("✅ SUCCESS!!")
} catch {
  case e: chisel3.internal.ChiselException => 
    println(s"❌ Chisel Elaboration Failed: ${e.getMessage}")
  case e: Throwable => 
    println(s"❌ Chisel Testing Failed: ${e.toString}")
}

## 7.2 Άσκηση
Θα υλοποιήσετε ένα νέο module `DataPath` το οποίο θα συνδυάζει το module `RegisterFile` (με `register_number` καταχωρητές και εύρος λέξης `register_width` bits) και το module `Alu` (με εύρος λέξης `register_width` bits).

**Η υλοποίηση πρέπει να γίνει σύμφωνα με το επόμενο σχήμα:**

![datapath1.png](https://mixstef.github.io/courses/comparch/labimg/datapath1.png)


Τα **νέα σήματα** στο σχήμα είναι το `alu_a_sel` και το `im` (για όλα τα υπόλοιπα σήματα ανατρέξτε στο αντίστοιχο εργαστήριο του `Alu` ή του `RegisterFile`).

Η λειτουργία του σήματος `alu_a_sel` (εύρους 1 bit) είναι η ακόλουθη:

* Όταν το `alu_a_sel` είναι 0, τότε στην είσοδο a της Alu εισάγεται η λέξη της εισόδου `im` (με εύρος `register_width` bits).

* Όταν το `alu_a_sel` είναι 1, τότε στην είσοδο a της Alu εισάγεται η έξοδος `out_a` του register file.

### 7.2.1 Οδηγίες συμπλήρωσης του module DataPath
Στο επόμενο κελί υπάρχει ο βασικός σκελετός κώδικα για το module `DataPath` για να συμπληρώσετε σύμφωνα με το προηγούμενο σχήμα. Παρατηρήστε πώς εισάγουμε instances των `RegisterFile` και `Alu` για χρήση μέσα στο module (**προσυμπληρωμένος κώδικας**):

~~~scala
val regfile = Module(new RegisterFile(register_number,register_width))
val alu = Module(new Alu(register_width))
~~~

#### Δημιουργία συνδέσεων
**α)** Συνδέστε τις εισόδους του κεντρικού module (`DataPath`) `read_sel_a`, `read_sel_b` και `write_sel` στις αντίστοιχες εισόδους του υπο-module `RegisterFile` (μεταβλητή `regfile`). Π.χ. για το `read_sel_a` θα γράψετε:

~~~scala
regfile.io.read_sel_a := io.read_sel_a
~~~

**β)** Συνδέστε με τον ίδιο τρόπο τις εισόδους του κεντρικού module (`DataPath`) `sel` και `sub` στις αντίστοιχες εισόδους του υπο-module `Alu` (μεταβλητή `alu`).

**γ)** Οδηγήστε την έξοδο `out` της `Alu` στην είσοδο `in` του `Registerfile`: 

~~~scala
regfile.io.in := alu.io.out
~~~

**δ)** Ομοίως, συνδέστε την είσοδο `b` της `Alu` με την έξοδο `out_b` του `RegisterFile`.

**ε)** Ο πολυπλέκτης (MUX) του σχήματος συνδέει την είσοδο `a` της `Alu`
* είτε με την είσοδο `im` (όταν `alu_a_sel` = 0)
* είτε με την έξοδο `out_a` του `RegisterFile` (όταν `alu_a_sel` = 0).

Συμπληρώστε την υπάρχουσα δομή `when` για να υλοποιήσετε τη λειτουργία αυτή (δηλ. το σύνολο των μωβ συνδέσεων στο σχήμα).

**στ)** Στο τέλος του κώδικα του module υπάρχει (**προσυμπληρωμένη**) η σύνδεση της εξόδου `out` της `Alu` με την έξοδο `results` του `DataPath`. Η έξοδος αυτή θα χρησιμοποιηθεί στη φάση αυτή για debugging, για να παρατηρούμε τα αποτελέσματα κατά την εκτέλεση των εντολών. 

In [ ]:
class DataPath(register_number: Int, register_width: Int) extends Module {
  val io = IO(new Bundle {
    // immediate value input
    val im = Input(UInt(register_width.W))
    // alu op selection inputs
    val sel = Input(UInt(2.W))
    val sub = Input(UInt(1.W))
    // immediate (0) or regfileA (1) selection input
    var alu_a_sel = Input(UInt(1.W))
    // regfile read/write register selection inputs
    val read_sel_a = Input(UInt(log2Ceil(register_number).W))
    val read_sel_b = Input(UInt(log2Ceil(register_number).W))
    val write_sel = Input(UInt(log2Ceil(register_number).W))
      
    // alu output for debugging
    val results = Output(UInt(register_width.W))
  })
    
  val regfile = Module(new RegisterFile(register_number,register_width))
  val alu = Module(new Alu(register_width))

  // (α)
  regfile.io.read_sel_a := io.read_sel_a
  // ...συμπληρώστε με όμοιο τρόπο για τα read_sel_b και write_sel...  

  // (β)
  // ...συμπληρώστε με όμοιο τρόπο για τα sel και sub...   

  // (γ)
  regfile.io.in := alu.io.out

  // (δ)
  // ...συμπληρώστε με όμοιο τρόπο τη σύνδεση μεταξύ b της Alu και out_b του RegisterFile...  

  // (ε)  
  when (io.alu_a_sel === 0.U){
    alu.io.a :=  // ...συμπληρώστε... 
  }.otherwise{
    alu.io.a :=  // ...συμπληρώστε... 
  }
     
  // (στ) debug only
  io.results := alu.io.out
}

## 7.3 Δοκιμή κυκλώματος datapath
Χρησιμοποιήστε τον κώδικα στο επόμενο κελί για να εκτελέσετε δύο δοκιμαστικές λειτουργίες (micro-instructions) σε ένα datapath με 8 καταχωρητές, με εύρος λέξης 16 bits.

*(Για την εξήγηση του κώδικα δείτε το κείμενο που ακολουθεί μετά τη δοκιμή)* 

In [ ]:
try {
  test(new DataPath(8,16)) { c =>
    // σήματα εισόδου για τις δοκιμαστικές "λειτουργίες" (micro-instructions)
    val cmdbits = List(Map("READ_SEL_A" -> 0.U,    // λειτουργία Α:  r1 = 333 or r0 (io.results = 333)
                         "READ_SEL_B" -> 0.U,
                         "ALU_A_SEL" -> 0.U,
                         "IM" -> 333.U,
                         "SEL" -> "b01".U,
                         "SUB" -> 0.U,
                         "WRITE_SEL" -> 1.U,
                         "RESULTS" -> 333.U),
                     Map("READ_SEL_A" -> 0.U,    // λειτουργία Β:  r0 = r0 or r1 (io.results = 333)
                         "READ_SEL_B" -> 1.U,
                         "ALU_A_SEL" -> 1.U,
                         "IM" -> 0.U,
                         "SEL" -> "b01".U,
                         "SUB" -> 0.U,
                         "WRITE_SEL" -> 0.U,
                         "RESULTS" -> 333.U)
                    )
  
    
    for (bits <- cmdbits) {
      c.io.read_sel_a.poke(bits("READ_SEL_A"))
      c.io.read_sel_b.poke(bits("READ_SEL_B"))
      c.io.alu_a_sel.poke(bits("ALU_A_SEL"))
      c.io.im.poke(bits("IM"))
      c.io.sel.poke(bits("SEL"))
      c.io.sub.poke(bits("SUB"))
      c.io.write_sel.poke(bits("WRITE_SEL"))
      println(c.io.results.peek())      
      c.io.results.expect(bits("RESULTS"))
      c.clock.step()
    }  
  }
  println("✅ SUCCESS!!")
} catch {
  case e: chisel3.internal.ChiselException => 
    println(s"❌ Chisel Elaboration Failed: ${e.getMessage}")
  case e: Throwable => 
    println(s"❌ Chisel Testing Failed: ${e.toString}")
}

### 7.3.1 Εξήγηση κώδικα Scala
Στον έλεγχο χρησιμοποιούμε δομές και τύπους της γλώσσας Scala ως εξής:

* Η μεταβλητή `cmdbits` είναι μια λίστα (**List**) από **Maps**.


* Στη Scala, **List** είναι μια ακολουθία που μπορούμε να διασχίσουμε από την μια άκρη στην άλλη (όχι όμως να πάρουμε το i-οστό στοιχείο). Στον κώδικα έχουμε ένα επαναληπτικό `for (bits <- cmdbits) {...}` που σε κάθε επανάληψη αναθέτει στη μεταβλητή `bits` ένα-ένα τα περιεχόμενα της λίστας `cmdbits`. Συνεπώς, η μεταβλητή `bits` σε κάθε επανάληψη του for είναι ένα διαφορετικό **Map**.


* **Map** στη Scala είναι μια δομή «λεξικού» (κλειδιού/τιμής). Στον κώδικα ελέγχου παίρνουμε τις τιμές από κάθε Map και οδηγούμε ανάλογα τα αντίστοιχα σήματα εισόδου του DataPath. Στη συνέχεια δίνουμε έναν παλμό ρολογιού και τυπώνουμε/ελέγχουμε την έξοδο `results`. Αυτό επαναλαμβάνεται για κάθε Map της λίστας.

Όπως φαίνεται και από τον αριθμό των Maps της λίστας `cmdbits`, στον κώδικα εκτελούνται 2 λειτουργίες (micro-instructions).

### 7.3.2 Δοκιμάστε κι εσείς
Στο επόμενο κελί συμπληρώστε το περιεχόμενο της λίστας `cmdbits` (θα έχει τρία Maps, όσα και οι ζητούμενες micro-instructions), έτσι ώστε να υλοποιηθούν οι εξής λειτουργίες:
~~~
r1 = 333 or r0        // το io.results πρέπει να είναι 333
r2 = 334 or r0        // το io.results πρέπει να είναι 334
r3 = r1 + r2          // το io.results πρέπει να είναι 667
~~~

**Θυμηθείτε ότι μπορείτε να τροποποιήσετε στον κώδικα μόνο το περιεχόμενο της λίστας `cmdbits`!**

In [ ]:
try {
  test(new DataPath(8,16)) { c =>
    // προσθέστε τρία Maps για τις ζητούμενες λειτουργίες
    val cmdbits = List(Map(...συμπληρώστε...),
                     Map(...συμπληρώστε...),
                     Map(...συμπληρώστε...)
                    )

    // μην αλλάξετε τον κώδικα από εδώ και κάτω!      
    
    for (bits <- cmdbits) {
      c.io.read_sel_a.poke(bits("READ_SEL_A"))
      c.io.read_sel_b.poke(bits("READ_SEL_B"))
      c.io.alu_a_sel.poke(bits("ALU_A_SEL"))
      c.io.im.poke(bits("IM"))
      c.io.sel.poke(bits("SEL"))
      c.io.sub.poke(bits("SUB"))
      c.io.write_sel.poke(bits("WRITE_SEL"))
      println(c.io.results.peek())
      c.io.results.expect(bits("RESULTS"))
      c.clock.step()
    }  
  }
  println("✅ SUCCESS!!")
} catch {
  case e: chisel3.internal.ChiselException => 
    println(s"❌ Chisel Elaboration Failed: ${e.getMessage}")
  case e: Throwable => 
    println(s"❌ Chisel Testing Failed: ${e.toString}")
}

## Τελειώνοντας...
Κατεβάσετε και πάρετε μαζί σας το σημερινό notebook, **είναι απαραίτητο για τα επόμενα εργαστήρια**).

**Δεν ανήκει στα παραδοτέα του εργαστηρίου.**